# AudioRestore Demo: Text-Controlled Audio Restoration

**CS 614 Final Project — Proof of Concept**

This notebook demonstrates SonicMaster fine-tuned for codec audio restoration.
Given a degraded audio file and a text prompt, the model restores audio quality.

## What this demo shows
1. Load the fine-tuned (or pretrained) SonicMaster model
2. Restore audio degraded at various bitrates using text prompts
3. Compare before/after spectrograms
4. Listen to before/after audio
5. Compute quality metrics (SDR, SI-SNR)

In [ ]:
# Cell 0: Detect environment & install deps
import os, sys

try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/CS614_Project'
    SM_DIR = '/content/sonicmaster'
    !pip install diffusers==0.30.0 transformers==4.44.0 accelerate==0.34.2 -q
    !pip install safetensors datasets librosa soundfile tqdm pandas pyyaml -q
    !pip install huggingface_hub -q
else:
    cwd = os.getcwd()
    if os.path.basename(cwd) == 'Project':
        PROJECT_ROOT = cwd
    elif os.path.isdir(os.path.join(cwd, 'Project')):
        PROJECT_ROOT = os.path.join(cwd, 'Project')
    else:
        PROJECT_ROOT = cwd
    SM_DIR = os.path.join(PROJECT_ROOT, 'sonicmaster')

print(f"Environment: {'Colab' if IS_COLAB else 'Local'}")
print(f"Project root: {PROJECT_ROOT}")
print(f"SonicMaster dir: {SM_DIR}")

In [ ]:
# Cell 1: Mount Drive (Colab) + clone SonicMaster + imports
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    print(f'Drive mounted. Project root: {PROJECT_ROOT}')

    if not os.path.exists(SM_DIR):
        !git clone https://github.com/AMAAI-Lab/SonicMaster.git {SM_DIR}
    sys.path.insert(0, SM_DIR)
    print(f'SonicMaster ready: {SM_DIR}')
else:
    if not os.path.isdir(SM_DIR):
        !git clone https://github.com/AMAAI-Lab/SonicMaster.git {SM_DIR}
    sys.path.insert(0, SM_DIR)

import torch
import torchaudio
import numpy as np
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display, HTML
import yaml

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# Cell 2: Load SonicMaster model + VAE
from model import TangoFlux
from safetensors.torch import load_file
from diffusers import AutoencoderOobleck

CONFIG_PATH = os.path.join(SM_DIR, 'configs', 'tangoflux_config.yaml')

if IS_COLAB:
    CKPT_PATH = os.path.join(PROJECT_ROOT, 'outputs', 'finetune_codec', 'best', 'model.safetensors')
else:
    CKPT_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', 'model.safetensors')

assert os.path.exists(CKPT_PATH), f'Checkpoint not found: {CKPT_PATH}'

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print('Loading TangoFlux model...')
model = TangoFlux(config=cfg['model'])
weights = load_file(CKPT_PATH)
model.load_state_dict(weights, strict=False)
model.to(device).eval()
for p in model.text_encoder.parameters():
    p.requires_grad = False
print(f'Model loaded from {CKPT_PATH}')

print('Loading Stable Audio VAE...')
# Set HF token: prefer Colab secrets, then env var, then fallback
hf_token = None
if IS_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
if not hf_token:
    print('No HF_TOKEN found. Add it as a Colab secret (key icon in sidebar).')
    print('  1. Accept license: https://huggingface.co/stabilityai/stable-audio-open-1.0')
    print('  2. Get token: https://huggingface.co/settings/tokens')
vae = AutoencoderOobleck.from_pretrained(
    'stabilityai/stable-audio-open-1.0', subfolder='vae',
    token=hf_token
).to(device).eval()
print('VAE loaded.')

In [ ]:
# Helper functions
SR = 44100
CHUNK_DUR = 30
CHUNK_SAMPLES = SR * CHUNK_DUR

def load_audio(path, sr=SR):
    """Load audio, force stereo, resample to target sr."""
    wav, orig_sr = torchaudio.load(path)
    if orig_sr != sr:
        wav = torchaudio.functional.resample(wav, orig_sr, sr)
    if wav.shape[0] == 1:
        wav = wav.repeat(2, 1)
    elif wav.shape[0] > 2:
        wav = wav[:2]
    return wav


def degrade_audio(wav, codec='mp3', bitrate=64000):
    """Apply codec degradation."""
    if codec == 'mp3':
        return torchaudio.functional.apply_codec(wav, SR, format='mp3', compression=bitrate//1000)
    else:
        return torchaudio.functional.apply_codec(wav, SR, format='ogg', compression=bitrate)


@torch.no_grad()
def restore_audio(wav_degraded, prompt, num_steps=10, guidance_scale=1.0):
    """Run SonicMaster restoration on a single chunk."""
    # Pad/trim to 30s
    T = wav_degraded.shape[1]
    if T < CHUNK_SAMPLES:
        wav_degraded = torch.nn.functional.pad(wav_degraded, (0, CHUNK_SAMPLES - T))
    elif T > CHUNK_SAMPLES:
        wav_degraded = wav_degraded[:, :CHUNK_SAMPLES]
    
    # Encode to latent
    z = vae.encode(wav_degraded.unsqueeze(0).to(device)).latent_dist.mode()  # [1, C, T']
    z = z.transpose(1, 2)  # [1, T', C]
    
    # Run flow matching inference
    result = model.inference_flow(
        z, prompt,
        audiocond_latents=None,
        num_inference_steps=num_steps,
        guidance_scale=guidance_scale,
        duration=CHUNK_DUR,
        seed=42,
        disable_progress=True,
    )
    
    # Decode back to waveform
    restored = vae.decode(result.transpose(1, 2)).sample  # [1, 2, T]
    restored = torch.clamp(restored.squeeze(0).cpu(), -1.0, 1.0)
    return restored[:, :T]  # Trim to original length


def plot_spectrograms(original, degraded, restored, sr=SR, title=''):
    """Plot three spectrograms side by side."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, wav, label in zip(axes, [original, degraded, restored],
                               ['Original', 'Degraded', 'Restored']):
        mono = wav[0].numpy() if wav.dim() == 2 else wav.numpy()
        S = librosa.feature.melspectrogram(y=mono, sr=sr, n_mels=128, fmax=sr//2)
        S_dB = librosa.power_to_db(S, ref=np.max)
        librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel', ax=ax, fmax=sr//2)
        ax.set_title(label, fontsize=13)
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def compute_sdr(reference, estimate):
    """Signal-to-Distortion Ratio in dB."""
    ref = reference[0].numpy() if reference.dim() == 2 else reference.numpy()
    est = estimate[0].numpy() if estimate.dim() == 2 else estimate.numpy()
    n = min(len(ref), len(est))
    ref, est = ref[:n], est[:n]
    noise = est - ref
    sdr = 10 * np.log10(np.sum(ref**2) / (np.sum(noise**2) + 1e-10))
    return sdr


def compute_sisnr(reference, estimate):
    """Scale-Invariant Signal-to-Noise Ratio in dB."""
    ref = reference[0].numpy() if reference.dim() == 2 else reference.numpy()
    est = estimate[0].numpy() if estimate.dim() == 2 else estimate.numpy()
    n = min(len(ref), len(est))
    ref, est = ref[:n], est[:n]
    ref = ref - np.mean(ref)
    est = est - np.mean(est)
    s_target = np.dot(est, ref) * ref / (np.dot(ref, ref) + 1e-10)
    e_noise = est - s_target
    return 10 * np.log10(np.sum(s_target**2) / (np.sum(e_noise**2) + 1e-10))

In [ ]:
# === DEMO: Restore a test audio file ===
import glob

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
CLEAN_DIR = os.path.join(DATA_DIR, 'clean')

# Search for a test audio file
candidates = []
for search_dir in [CLEAN_DIR, DATA_DIR, os.path.join(DATA_DIR, 'clean_clips')]:
    if os.path.isdir(search_dir):
        candidates += glob.glob(os.path.join(search_dir, '*.wav'))
        candidates += glob.glob(os.path.join(search_dir, '*.flac'))

if IS_COLAB and not candidates:
    print('No audio found on Drive. Upload a WAV file:')
    from google.colab import files
    uploaded = files.upload()
    TEST_AUDIO = list(uploaded.keys())[0]
elif candidates:
    TEST_AUDIO = candidates[0]
else:
    raise FileNotFoundError(f'No audio files found. Place a .wav or .flac in {CLEAN_DIR}')

print(f'Test audio: {TEST_AUDIO}')
original = load_audio(TEST_AUDIO)
if original.shape[1] > CHUNK_SAMPLES:
    original = original[:, :CHUNK_SAMPLES]
print(f'Shape: {original.shape}, Duration: {original.shape[1]/SR:.1f}s')

In [ ]:
# === Test at multiple bitrates ===

bitrates = [32000, 64000, 128000]
results = {}

for br in bitrates:
    br_k = br // 1000
    print(f'\n--- {br_k} kbps MP3 ---')
    
    degraded = degrade_audio(original, 'mp3', br)
    # Match lengths
    n = min(original.shape[1], degraded.shape[1])
    degraded = degraded[:, :n]
    orig_trim = original[:, :n]
    
    prompt = f'restore audio compressed at {br_k}kbps MP3'
    print(f'  Prompt: "{prompt}"')
    print(f'  Running restoration...')
    restored = restore_audio(degraded, prompt)
    restored = restored[:, :n]
    
    sdr_deg = compute_sdr(orig_trim, degraded)
    sdr_res = compute_sdr(orig_trim, restored)
    sisnr_deg = compute_sisnr(orig_trim, degraded)
    sisnr_res = compute_sisnr(orig_trim, restored)
    
    results[br_k] = {
        'sdr_degraded': sdr_deg, 'sdr_restored': sdr_res,
        'sisnr_degraded': sisnr_deg, 'sisnr_restored': sisnr_res,
    }
    
    print(f'  SDR:    {sdr_deg:.2f} dB -> {sdr_res:.2f} dB (improvement: {sdr_res-sdr_deg:+.2f} dB)')
    print(f'  SI-SNR: {sisnr_deg:.2f} dB -> {sisnr_res:.2f} dB (improvement: {sisnr_res-sisnr_deg:+.2f} dB)')
    
    plot_spectrograms(orig_trim, degraded, restored, title=f'MP3 @ {br_k} kbps')
    
    print('  Original:')
    display(Audio(orig_trim[0].numpy(), rate=SR))
    print(f'  Degraded ({br_k} kbps):')
    display(Audio(degraded[0].numpy(), rate=SR))
    print('  Restored:')
    display(Audio(restored[0].numpy(), rate=SR))

In [ ]:
# === Summary metrics table ===

print('\n' + '='*70)
print(f'{"Bitrate":>10} | {"SDR Degraded":>14} | {"SDR Restored":>14} | {"SDR Gain":>10}')
print(f'{"":>10} | {"SI-SNR Deg":>14} | {"SI-SNR Res":>14} | {"SI-SNR Gain":>10}')
print('-'*70)
for br_k, m in results.items():
    print(f'{br_k:>8} k | {m["sdr_degraded"]:>12.2f} dB | {m["sdr_restored"]:>12.2f} dB | {m["sdr_restored"]-m["sdr_degraded"]:>+8.2f} dB')
    print(f'{"":>10} | {m["sisnr_degraded"]:>12.2f} dB | {m["sisnr_restored"]:>12.2f} dB | {m["sisnr_restored"]-m["sisnr_degraded"]:>+8.2f} dB')
    print('-'*70)
print('='*70)

In [ ]:
# === Custom prompt restoration ===
# Try different prompts on the same degraded audio

degraded_64k = degrade_audio(original[:, :CHUNK_SAMPLES], 'mp3', 64000)

prompts_to_test = [
    'remove MP3 compression artifacts',
    'restore audio compressed at 64kbps MP3',
    'enhance low bitrate audio to high fidelity',
    'make this audio sound better',
    '',  # empty prompt (unconditional)
]

print('Degraded audio: 64 kbps MP3')
print('Testing different text prompts:\n')

for prompt in prompts_to_test:
    label = prompt if prompt else '(empty / unconditional)'
    print(f'Prompt: "{label}"')
    restored = restore_audio(degraded_64k, prompt)
    n = min(original.shape[1], restored.shape[1])
    sdr = compute_sdr(original[:, :n], restored[:, :n])
    print(f'  SDR: {sdr:.2f} dB')
    display(Audio(restored[0, :n].numpy(), rate=SR))
    print()